# tlakit quickstart

Model-check a TLA+ spec from a notebook. Requires Java and
`TLAKIT_TLA2TOOLS` pointing at `tla2tools.jar`.


In [ ]:
%load_ext tlakit

## Define a module

A microwave whose door can be opened while the radiation is on.


In [ ]:
%%tla Microwave
---- MODULE Microwave ----
EXTENDS Naturals

VARIABLES door, radiation, timeRemaining
vars == <<door, radiation, timeRemaining>>

Init == door = "closed" /\ radiation = "off" /\ timeRemaining = 0

\* Opening the door does not stop the radiation. That is the bug.
Open    == door' = "open"   /\ UNCHANGED <<radiation, timeRemaining>>
Close   == door' = "closed" /\ UNCHANGED <<radiation, timeRemaining>>
AddTime == timeRemaining < 2 /\ timeRemaining' = timeRemaining + 1
                             /\ UNCHANGED <<door, radiation>>
Start   == timeRemaining > 0 /\ radiation' = "on"
                             /\ UNCHANGED <<door, timeRemaining>>
Tick    == radiation = "on" /\ timeRemaining > 0
                            /\ timeRemaining' = timeRemaining - 1
                            /\ UNCHANGED <<door, radiation>>

Next == Open \/ Close \/ AddTime \/ Start \/ Tick
Spec == Init /\ [][Next]_vars

Safety == radiation = "on" => door = "closed"
====


## Check the safety invariant

The cell body is the TLC configuration file.


In [ ]:
%%tlc Microwave
SPECIFICATION Spec
INVARIANT Safety


## Work with the counterexample in Python


In [ ]:
import tlakit
from tlakit.magics import MODULES   # modules defined by %%tla live here

spec = tlakit.Spec(source=MODULES["Microwave"], name="Microwave")
result = spec.check(invariants=["Safety"])

print(result.outcome)
print("steps:", len(result.trace))
for i, state in enumerate(result.trace.states):
    changed = sorted(result.trace.delta(i))
    print(f"{i + 1}: changed={changed} {state}")

assert result.outcome is tlakit.Outcome.INVARIANT_VIOLATION
